# Chapter 04 · 지능형 RAG 시스템 구축 (Building Intelligent RAG Systems)

---

### 📑 목차

| 섹션 | 주제 | 핵심 키워드 |
|------|------|------------|
| **0** | 환경 설정 & 샘플 데이터 | API Key · knowledge_base.json |
| **1** | RAG 기초 | 4개 구성요소 · 2개 파이프라인 · 도입 기준 |
| **2** | 임베딩 & 벡터스토어 | 1536-dim · Chroma · 인덱싱(HNSW/IVF) |
| **3** | RAG 파이프라인 | Loader → Chunking(5단계) → Retriever |
| **4** | Advanced RAG | Hybrid · Re-rank · HyDE · MMR · 인용 · CRAG · Agentic |
| **5** | 실전 & 트러블슈팅 | Corporate Chatbot · 7 Failure Points |

> ⚠️ **실행 안내**: 대부분의 셀은 OpenAI API 키가 필요합니다(임베딩·LLM 호출).
> 키 없이도 읽으며 구조를 파악할 수 있도록, 각 셀은 독립적으로 구성하고 핵심 개념은 마크다운으로 함께 정리했습니다.

---
## SECTION 0 · 환경 설정 & 샘플 데이터

RAG 실습에 필요한 패키지를 설치하고, 모든 섹션에서 공통으로 쓰는 샘플 지식베이스(`knowledge_base.json`)를 생성합니다.

In [ ]:
# --- (1) 패키지 설치 -------------------------------------------------------
# 필요 시 주석을 풀어 실행하세요.  
%pip install -qU langchain langchain-openai langchain-community \
     langchain-text-splitters langchain-experimental langchain-chroma \
     faiss-cpu rank_bm25 jq tiktoken

In [ ]:
# --- (2) API 키 설정 -------------------------------------------------------
# setting the environment variables, the keys
from dotenv import load_dotenv
import sys
import os

load_dotenv()

### 샘플 지식베이스 생성

원서 `chapter4/knowledge_base.json`과 동일한 형식입니다. Transformer · BERT · GPT · 기후변화 등
AI/일반 주제 문서를 담고 있으며, 이후 모든 검색 예제의 코퍼스로 사용합니다.

In [ ]:
# --- (3) knowledge_base.json 생성 -----------------------------------------
import json

knowledge_base = [
    {"content": "Transformer models were introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. The architecture relies on self-attention mechanisms rather than recurrent or convolutional neural networks. This design allows for more parallelization during training and better handling of long-range dependencies in text.",
     "metadata": {"source": "AI_Textbook.pdf", "page": 42, "topic": "transformer_architecture"}},
    {"content": "BERT (Bidirectional Encoder Representations from Transformers) was developed by Google AI Language team in 2018. It is pre-trained using masked language modeling and next sentence prediction tasks. BERT is designed to pre-train deep bidirectional representations by jointly conditioning on both left and right context in all layers.",
     "metadata": {"source": "NLP_Models_Survey.pdf", "page": 137, "topic": "bert_model"}},
    {"content": "GPT (Generative Pre-trained Transformer) models are autoregressive language models that use transformer-based neural networks. Unlike BERT, which is bidirectional, GPT models are unidirectional and predict the next token based on previous tokens. The original GPT was introduced by OpenAI in 2018, followed by GPT-2 in 2019 and GPT-3 in 2020.",
     "metadata": {"source": "Large_Language_Models.pdf", "page": 89, "topic": "gpt_models"}},
    {"content": "Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be natural, but since the 1800s, human activities have been the main driver of climate change, primarily due to the burning of fossil fuels like coal, oil and gas, which produces heat-trapping greenhouse gases.",
     "metadata": {"source": "Climate_Report.pdf", "page": 5, "topic": "climate_change"}},
    {"content": "Reducing your carbon footprint through dietary changes can be significant. Plant-based diets generally have lower greenhouse gas emissions than diets rich in animal products. Beef production, in particular, generates substantial methane emissions and requires large amounts of land and water resources.",
     "metadata": {"source": "Climate_Report.pdf", "page": 12, "topic": "carbon_footprint"}},
    {"content": "Retrieval-Augmented Generation (RAG) combines retrieval systems with generative AI models. It helps address hallucinations by grounding responses in retrieved information, enabling models to access up-to-date and domain-specific knowledge without retraining.",
     "metadata": {"source": "RAG_Survey.pdf", "page": 1, "topic": "rag_overview"}},
]

with open("knowledge_base.json", "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)

print(f"Wrote knowledge_base.json with {len(knowledge_base)} documents")

---
## SECTION 1 · RAG 기초 (From Indexes to Intelligent Retrieval)

### 왜 RAG인가?
70년 정보검색의 진화(AltaVista 1996 → Google PageRank 1998 → Word2Vec 2013 → BERT 2018 → **RAG 2020+**)는
"키워드 매칭은 *의미*를 포착하지 못한다"(turtle ≠ tortoise)는 한계로 수렴합니다.
**임베딩**이 "의미 지도"를 만들어 이 격차를 해소하고, **RAG**는 *검색 + 생성*을 결합합니다.

### RAG 4개 구성요소 + 2개 파이프라인

| 구성요소 | 역할 |
|----------|------|
| 📚 **Knowledge Base** | 외부 정보 저장 계층 |
| 🔍 **Retriever** | 관련 정보 검색 계층 |
| 🧩 **Augmenter** | 검색 결과 통합(프롬프트 결합) 계층 |
| ✍ **Generator** | 최종 응답 생성(LLM) 계층 |

- **Indexing Pipeline** (오프라인): 문서 처리 → chunk 분할 → 임베딩 → 벡터스토어 적재
- **Query Pipeline** (온라인): 쿼리 → 검색 → augmentation → LLM 생성 → 응답

### 언제 RAG을 도입할까

| ✅ RAG이 빛나는 곳 | ⚠️ Pure LLM도 충분한 경우 |
|---|---|
| 학습 데이터에 없는 최신 정보 | 창작·일반 대화 중심 작업 |
| 도메인 특화 지식(의료·금융·법률) | 초저지연 요구 |
| 출처 명시 검증 가능한 응답 | 정적/제한된 KB → fine-tuning이 효율적 |
| 규제 산업의 높은 정밀도 | 프로토타입·POC 단계 |

아래 셀은 "검색 → 증강 → 생성"의 **최소 RAG 루프**를 한눈에 보여줍니다.

In [ ]:
# 가장 단순한 RAG 루프 — 4개 구성요소를 모두 한 번에 (Naive RAG)
from langchain_community.document_loaders import JSONLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 📚 Knowledge Base
loader = JSONLoader(file_path="knowledge_base.json", jq_schema=".[].content", text_content=True)
documents = loader.load()

# 🔍 Retriever (임베딩 + 벡터스토어)
embedder = OpenAIEmbeddings()
vector_db = FAISS.from_documents(documents, embedder)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 🧩 Augmenter (검색 결과를 프롬프트에 결합) + ✍ Generator
prompt = ChatPromptTemplate.from_template(
    "Answer the question based ONLY on the context below.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

question = "What are the effects of climate change?"
retrieved = retriever.invoke(question)
answer = (prompt | llm | StrOutputParser()).invoke(
    {"context": format_docs(retrieved), "question": question}
)
print(answer)

---
## SECTION 2 · 임베딩 & 벡터스토어 (From Embeddings to Search)

### 2.1 임베딩 (Embeddings)
텍스트를 의미 공간의 **dense vector**로 변환합니다. OpenAI 기준 1536차원.
- `embed_documents`: 여러 문서 임베딩 (인덱싱 시점)
- `embed_query`: 단일 쿼리 임베딩
- 의미가 비슷한 문장은 코사인 유사도가 높습니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings()

text1 = "The cat sat on the mat"
text2 = "A feline rested on the carpet"   # text1과 의미 유사
text3 = "Python is a programming language"  # 무관

embeddings = embeddings_model.embed_documents([text1, text2, text3])
print(f"Number of documents: {len(embeddings)}")
print(f"Dimensions per embedding: {len(embeddings[0])}")  # 보통 1536

In [ ]:
# 코사인 유사도로 "의미 거리" 직접 확인
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"sim(cat,  feline) = {cosine(embeddings[0], embeddings[1]):.3f}  # 높음")
print(f"sim(cat,  python) = {cosine(embeddings[0], embeddings[2]):.3f}  # 낮음")

### 2.2 벡터스토어 비교 (Table 4.1)

| DB | 배포 | 라이선스 | 핵심 특징 |
|----|------|----------|-----------|
| Pinecone | Cloud only | Commercial | Auto-scaling · 엔터프라이즈 보안 |
| Milvus | Cloud/Self | Apache 2.0 | HNSW/IVF · 멀티모달 |
| Weaviate | Cloud/Self | BSD 3-Clause | 그래프 구조 · 멀티모달 |
| Qdrant | Cloud/Self | Apache 2.0 | HNSW · 필터링 최적화 |
| **ChromaDB** | Cloud/Self | Apache 2.0 | 경량 · 빠른 셋업 |
| pg_vector | Cloud/Self | OSS | SQL · Postgres 통합 |
| Vertex VS | Cloud only | Commercial | 낮은 지연 · 고가용성 |

> **선택 가이드**: 시작은 **ChromaDB** · 엔터프라이즈는 **Pinecone** · 기존 Postgres 인프라엔 **pg_vector**

### 2.3 LangChain Vector Store 인터페이스
통일된 API로 구현체를 자유롭게 교체할 수 있습니다.
`add_documents` · `similarity_search` · `max_marginal_relevance_search` · `delete(ids)`

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()

# 명시적 ID를 가진 샘플 문서
docs = [
    Document(page_content="Content about language models",      metadata={"id": "doc_1"}),
    Document(page_content="Information about vector databases", metadata={"id": "doc_2"}),
    Document(page_content="Details about retrieval systems",    metadata={"id": "doc_3"}),
]

vector_store = Chroma(embedding_function=embeddings)

# 1. 문서 추가 (+ 자동 임베딩)
vector_store.add_documents(docs)

# 2. 유사도 검색 (Top-k)
results = vector_store.similarity_search("How do language models work?", k=2)
print("similarity_search:", [d.page_content for d in results])

# 3. MMR — 관련성과 다양성 균형
found_docs = vector_store.similarity_search("retrieval", k=1)
used_ids = {d.metadata["id"] for d in found_docs}
remaining = [d for d in docs if d.metadata["id"] not in used_ids]
print(f"Remaining for MMR: {len(remaining)}")

### 2.4 벡터 인덱싱 전략 (Table 4.2)
Brute force는 O(D·N) — 대규모에서는 인덱스가 필수입니다.

| 전략 | 알고리즘 | 검색 복잡도 | 메모리 | 적합 |
|------|----------|------------|--------|------|
| **Exact (Flat)** | 전수 비교 | O(D·N) | Low | <100K · 100% recall · 베이스라인 |
| **HNSW** | 계층형 그래프 | O(log N) | High | 프로덕션 표준 · 대규모 |
| **LSH** | 해시 버킷팅 | O(N) | Med | 스트리밍 · 빈번한 업데이트 |
| **IVF** | 클러스터 분할 | O(N/k) | Med | 대규모 + 메모리 절감 |
| **PQ** | 벡터 양자화 | — | 매우 Low | IVF+PQ 조합으로 메모리 극단 절감 |

> 실무 결론: **HNSW가 프로덕션 표준**, 메모리가 빠듯하면 **IVF + PQ** 조합.

---
## SECTION 3 · RAG 파이프라인 (Loader → Chunking → Retrieval)

### 도서관 비유: 책 정리부터 대출까지
1. **Document Processing** — 다양한 포맷 로드 → 표준 `Document`로 변환 → 의미있는 chunk 분할
2. **Vector Indexing** — 각 chunk를 임베딩으로 변환
3. **Vector Storage** — 벡터 + 원본 텍스트 + 메타데이터 저장 (HNSW/IVF)
4. **Retrieval** — 쿼리를 동일 모델로 임베딩 → Top-k 유사 문서 반환

### 3.1 Document Loaders (Table 4.3)
| 카테고리 | 로더 예시 |
|----------|-----------|
| File Systems | TextLoader · CSVLoader · PyPDFLoader |
| Web Content | WebBaseLoader · RecursiveURLLoader · SitemapLoader |
| Cloud Storage | S3DirectoryLoader · GCSFileLoader · Dropbox |
| Databases | MongoDBLoader · SnowflakeLoader · BigQueryLoader |
| Productivity | NotionDirectoryLoader · SlackDirectory · Trello |
| Scientific | ArxivLoader · PubMedLoader |

In [ ]:
from langchain_community.document_loaders import JSONLoader

# JSON 파일에서 각 항목의 content 필드만 추출
loader = JSONLoader(
    file_path="knowledge_base.json",
    jq_schema=".[].content",   # 배열의 각 항목에서 content 추출
    text_content=True,
)
documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(documents[0].page_content[:120], "...")

### 3.2 Chunking 전략 5단계 (단순 → 정교)

| 레벨 | 전략 | 도구 | 특징 |
|------|------|------|------|
| **L1** | Fixed-size | `CharacterTextSplitter` | 빠르지만 문장 중간 잘림 |
| **L2** ⭐ | Recursive | `RecursiveCharacterTextSplitter` | 구분자 우선순위 재귀 분할 · **권장 기본값** |
| **L3** | Document-specific | `MarkdownSplitter` · `HTMLHeaderSplitter` | 문서 구조 보존 |
| **L4** | Semantic | `SemanticChunker` | 임베딩 유사도로 자연 경계 식별 (비용↑) |
| **L5** | Agent-based | LLM이 직접 분할 위치 결정 | 최고 비용 · 매우 복잡한 문서용 |

> **권장**: `RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)`

In [ ]:
# L1 — Fixed-Size Chunking
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator=" ",      # 단어가 깨지지 않도록 공백 기준
    chunk_size=200,
    chunk_overlap=20,
)
chunks = text_splitter.split_documents(documents)
print(f"[L1 Fixed] Generated {len(chunks)} chunks")

In [ ]:
# L2 — Recursive Character Chunking (권장 기본값)
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],   # 우선순위: 문단 > 줄 > 문장 > 단어
    chunk_size=150,
    chunk_overlap=20,
)

document = """# Introduction to RAG
Retrieval-Augmented Generation (RAG) combines retrieval systems with generative AI models.

It helps address hallucinations by grounding responses in retrieved information.

## Key Components
RAG consists of several components:
1. Document processing
2. Vector embedding
3. Retrieval
4. Augmentation
5. Generation
"""

chunks = text_splitter.split_text(document)
print(f"[L2 Recursive] {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ---\n{c}\n")

In [ ]:
# L4 — Semantic Chunking (임베딩 유사도로 자연스러운 경계 식별)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    add_start_index=True,   # 위치 메타데이터 포함
)
chunks = semantic_splitter.split_text(document)
print(f"[L4 Semantic] {len(chunks)} chunks")
for c in chunks:
    print("-", c[:80])

### 3.3 Retriever 종류
`as_retriever()` 외에도 다양한 retriever를 제공합니다. (KNN, PubMed 등 외부 검색 API 포함)

In [ ]:
# KNN Retriever — 임베딩 기반 k-최근접
from langchain_community.retrievers import KNNRetriever
from langchain_openai import OpenAIEmbeddings

retriever = KNNRetriever.from_documents(documents, OpenAIEmbeddings())
results = retriever.invoke("How do transformer models work?")
print(f"KNN retrieved {len(results)} docs")
print(results[0].page_content[:120], "...")

In [ ]:
# 외부 검색 API Retriever 예시 (벡터스토어 불필요) — 실행 시 네트워크 필요
# from langchain_community.retrievers.pubmed import PubMedRetriever
# retriever = PubMedRetriever()
# results = retriever.invoke("COVID research")
# print(results[0].page_content[:200])
print("PubMedRetriever: 외부 API 검색 retriever 예시 (위 주석 해제 후 실행)")

---
## SECTION 4 · Advanced RAG (Hybrid · Re-rank · HyDE · CRAG · Agentic)

Naive RAG의 한계를 보완하는 고급 기법들입니다. **요구사항에 맞춰 점진적으로** 추가하세요.

### 기법 선택 가이드 (Table 4.5)
| 기법 | 메커니즘 | 복잡도 | 적합 |
|------|----------|--------|------|
| Naive RAG | 단순 검색→생성 | Low | 프로토타입 |
| Hybrid Retrieval | BM25 + Vector | Medium | 기술 문서 · 전문 용어 |
| Re-ranking | Cohere/RankLLM | Medium | 검색 품질 정제 |
| Query Transform | HyDE · expansion | Medium | 모호한 쿼리 |
| Context Processing | Compression · MMR | Medium | 긴 문서 · 중복 |
| Response Enhancement | 인용 · 검증 | Med-High | 법률/의료 |
| CRAG | 평가 + 조건부 교정 | High | 고신뢰 요구 |
| Agentic RAG | 자율 오케스트레이션 | Very High | 복잡 다단계 |

### 4.1 Hybrid Retrieval — Dense(의미) + Sparse(키워드)
- **Dense**: 벡터 임베딩 → 동의어·패러프레이즈에 강함
- **Sparse**: BM25/TF-IDF → 전문 용어·고유명사에 강함
- **Weighted Fusion**: 가중치(예: 0.7/0.3)로 균형 조절

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# Dense — 의미 기반 (앞서 만든 vector_db 재사용)
vec_ret = vector_db.as_retriever(search_kwargs={"k": 5})

# Sparse — 키워드 정밀도
bm25 = BM25Retriever.from_documents(documents)
bm25.k = 5

# Rank fusion으로 결합 (0.7=의미 우선, 0.3=키워드)
hybrid = EnsembleRetriever(retrievers=[vec_ret, bm25], weights=[0.7, 0.3])

results = hybrid.invoke("What did Vaswani et al. introduce in 2017?")
print(f"Hybrid retrieved {len(results)} docs")
print(results[0].page_content[:140], "...")

### 4.2 Re-ranking — 초기 검색 후 정교 모델로 재정렬
- **Pointwise**: 각 문서를 독립 점수화 후 정렬
- **Pairwise**: 문서 쌍 비교 → 승패로 순위
- **Listwise**: 전체 리스트 동시 처리 (NDCG·MAP 최적화)

> 아래는 Cohere Rerank 예시입니다. `COHERE_API_KEY`가 필요합니다.

In [ ]:
# Re-ranking — ContextualCompressionRetriever + CohereRerank
# 실행하려면 COHERE_API_KEY 설정 + `pip install langchain-cohere` 필요
"""
from langchain.retrievers.document_compressors import CohereRerank
from langchain.retrievers import ContextualCompressionRetriever

base_retriever = vector_db.as_retriever(search_kwargs={"k": 10})
compressor = CohereRerank(top_n=3)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
reranked = compression_retriever.invoke("How do transformers work?")
"""
print("Re-ranking: 위 코드 블록을 해제하고 COHERE_API_KEY 설정 후 실행하세요.")

OpenAI로도 LLM 기반 재순위는 충분히 가능합니다. 채팅 모델에게 문서들을 보여주고 관련도 순으로 정렬/필터링하게 하는 방식이고, LangChain에 이미 압축기(compressor)로 구현

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

base_retriever = vector_db.as_retriever(search_kwargs={"k": 10})
compressor = LLMListwiseRerank.from_llm(llm, top_n=3)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
reranked = compression_retriever.invoke("How do transformers work?")

### 4.3 Query Transformation — 어휘 격차 해소
- **Query Expansion**: LLM이 같은 의도를 다양한 표현으로 재생성
- **HyDE** (Hypothetical Document Embeddings): 쿼리 임베딩과 답변 문서 임베딩은 의미가 다를 수 있음(질문 vs 진술).
  LLM이 만든 *가상 답변*으로 검색하면 실제 답변 문서를 더 잘 찾습니다.

In [ ]:
# Query Expansion
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

expansion_template = """Given the user question: {question}
Generate three alternative versions that express the same information need but with different wording:
1."""

expansion_prompt = PromptTemplate(input_variables=["question"], template=expansion_template)
llm = ChatOpenAI(temperature=0.7)
expansion_chain = expansion_prompt | llm | StrOutputParser()

print(expansion_chain.invoke("What are the effects of climate change?"))

In [ ]:
# HyDE — Hypothetical Document Embeddings
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

hyde_template = """Based on the question: {question}
Write a passage that could contain the answer to this question:"""
hyde_prompt = PromptTemplate(input_variables=["question"], template=hyde_template)

llm = ChatOpenAI(temperature=0.2)
hyde_chain = hyde_prompt | llm | StrOutputParser()

query = "What dietary changes can reduce carbon footprint?"
hypothetical_doc = hyde_chain.invoke(query)          # 1) 가상 답변 문서 생성
print("Hypothetical doc:\n", hypothetical_doc[:200], "...\n")

embeddings = OpenAIEmbeddings()
embedded_query = embeddings.embed_query(hypothetical_doc)   # 2) 그 문서를 임베딩
results = vector_db.similarity_search_by_vector(embedded_query, k=3)  # 3) 벡터로 검색
print("Top result:", results[0].page_content[:140], "...")

### 4.4 Context Processing — 압축 + 다양성
- **Contextual Compression**: 검색된 문서에서 *관련 부분만* 추출 (LLMChainExtractor)
- **MMR (Maximum Marginal Relevance)**: 관련성 ↔ 다양성 균형 (`lambda_mult`: 0=다양성 최대, 1=관련성 최대)

In [ ]:
# Contextual Compression — 관련 문장만 추출
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)
compressor = LLMChainExtractor.from_llm(llm)
base_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
compressed_docs = compression_retriever.invoke("How do transformer models work?")
print(f"Compressed to {len(compressed_docs)} focused snippets")
for d in compressed_docs:
    print("-", d.page_content[:100])

In [ ]:
# MMR — 관련성과 다양성 균형
mmr_results = vector_db.max_marginal_relevance_search(
    query="What are transformer models?",
    k=5,             # 반환 문서 수
    fetch_k=20,      # 초기 후보 수
    lambda_mult=0.5, # 0 = 다양성 최대, 1 = 관련성 최대
)
print(f"MMR returned {len(mmr_results)} diverse docs")
for d in mmr_results:
    print("-", d.page_content[:90])

### 4.5 Response Enhancement — 인용 + 자가 검증
- **Source Attribution**: `[1] [2]` 형식 인용 + 참고문헌 → 사용자가 사실 검증 가능 (의료·법률·학술 필수)
- **Self-Consistency Check**: 각 주장을 검색 컨텍스트와 대조 →
  `fully_supported` / `partial` / `contradicted` / `not_mentioned` 4단계 분류 (JSON 구조로 후처리)

In [ ]:
# Source Attribution — 인용이 포함된 답변 생성
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

cited_documents = [
    Document(page_content="The transformer architecture was introduced in the paper 'Attention is All You Need' by Vaswani et al. in 2017.",
             metadata={"source": "Neural Network Review 2021", "page": 42}),
    Document(page_content="BERT uses bidirectional training of the Transformer, masked language modeling, and next sentence prediction.",
             metadata={"source": "Introduction to NLP", "page": 137}),
    Document(page_content="GPT models are autoregressive transformers that predict the next token based on previous tokens.",
             metadata={"source": "Large Language Models Survey", "page": 89}),
]

emb = OpenAIEmbeddings()
store = FAISS.from_documents(cited_documents, emb)
retriever = store.as_retriever(search_kwargs={"k": 3})

attribution_prompt = ChatPromptTemplate.from_template("""
You are a precise AI assistant. Answer the question based ONLY on the provided sources.
For each fact, include a citation [1], [2], etc., and list the references at the end.

Sources:
{sources}

Question: {question}
""")

def format_sources(docs):
    return "\n".join(f"[{i+1}] {d.page_content} (Source: {d.metadata['source']}, p.{d.metadata['page']})"
                     for i, d in enumerate(docs))

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
q = "Who introduced the transformer architecture and what is BERT?"
docs = retriever.invoke(q)
answer = (attribution_prompt | llm | StrOutputParser()).invoke(
    {"sources": format_sources(docs), "question": q}
)
print(answer)

In [ ]:
# Self-Consistency Check — 생성된 답변이 컨텍스트로 뒷받침되는지 검증
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from typing import List
from langchain_core.documents import Document

def verify_response_accuracy(retrieved_docs: List[Document],
                             generated_answer: str,
                             llm: ChatOpenAI = None) -> str:
    """답변이 검색 문서로 완전히 뒷받침되는지 검증한다."""
    if llm is None:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    context = "\n\n".join(d.page_content for d in retrieved_docs)
    verification_prompt = ChatPromptTemplate.from_template("""
As a fact-checking assistant, verify whether the answer is fully supported by the context.
For each claim, classify as: fully_supported / partial / contradicted / not_mentioned.

Context:
{context}

Answer to verify:
{answer}

Return your analysis:""")

    return (verification_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "answer": generated_answer}
    )

report = verify_response_accuracy(docs, answer)
print(report)

### 4.6 Corrective RAG (CRAG) — 검색 결과를 평가하고 조건부 교정
1. **Initial Retrieval** — 표준 RAG 검색
2. **Retrieval Evaluator** — LLM이 각 문서 관련성 평가
3. **Conditional Branch** — Relevant / Irrelevant / Ambiguous
   - ✓ Relevant → 그대로 통과
   - ✗ Irrelevant → 필터링 (노이즈 제거)
   - ? Ambiguous → Web Search 등 대안적 정보 수집 트리거
4. **Generation** — 필터된 컨텍스트로 응답

아래는 핵심 아이디어(평가 + 조건부 필터링)를 보여주는 간소화 구현입니다.

In [ ]:
# CRAG (간소화) — 문서별 관련성 평가 후 필터링
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
grade_prompt = ChatPromptTemplate.from_template(
    "Is the following document relevant to the question? Answer only 'yes' or 'no'.\n\n"
    "Question: {question}\nDocument: {document}"
)
grader = grade_prompt | eval_llm | StrOutputParser()

def corrective_rag(question, k=4):
    candidates = vector_db.similarity_search(question, k=k)
    relevant = []
    for d in candidates:
        verdict = grader.invoke({"question": question, "document": d.page_content}).strip().lower()
        print(f"[{verdict:>3}] {d.page_content[:70]}...")
        if verdict.startswith("yes"):
            relevant.append(d)
    if not relevant:
        print(">> 관련 문서 없음 → Web Search 등 대안 정보 수집 트리거 (Ambiguous)")
    return relevant

filtered = corrective_rag("What are the effects of climate change?")
print(f"\nKept {len(filtered)} relevant docs after correction")

### 4.7 Agentic RAG — 자율 에이전트가 RAG 전체를 오케스트레이션
- **Query Decomposition** — 복잡한 질문을 하위 질문으로 분해
- **Information Planning** — 작업별 정보 수집 전략
- **Tool Selection** — Retriever · Web Search · Calculator · API
- **Multi-step Execution** — 여러 라운드 검색 + 추론
- **Reflection** — 중간 결과 평가 후 전략 적응

> **CRAG vs Agentic**: CRAG는 *데이터 품질*에 초점, Agentic은 *프로세스 지능*에 초점.
> 구현은 LangGraph 기반이며 Chapter 5(Agents)에서 본격적으로 다룹니다.

---
## SECTION 5 · 실전 & 트러블슈팅 (Practice & Troubleshooting)

### 5.1 Corporate Documentation Chatbot
원서 `streamlit_app.py` / `rag.py`의 완성형 파이프라인 — LangChain(LLM) + LangGraph(상태) + Streamlit(UI).

| 구성 | 내용 |
|------|------|
| 📥 Document Loading | PDF·TXT·EPUB·DOCX 로더 매핑 → `load_document(path)` |
| 🧠 Language Model | ChatGroq(deepseek-r1-70b) 생성 + 수정, OpenAIEmbeddings(+캐시) |
| 🔍 Retriever | InMemoryVectorStore + Recursive Splitter (1500/200) |
| 🕸 State Graph | retrieve → generate → double_check → doc_finalizer |
| 🖥 Streamlit UI | 채팅창 + 파일 업로더, 2-column 레이아웃 |
| ✓ Compliance Loop | LLM이 'ISSUES FOUND' 시 수정본 생성 (HITL 확장 가능) |

### State Graph 흐름 (LangGraph)
`retrieve`(Top-k 검색) → `generate`(초안) → `double_check`(기업 표준 준수 검토) → `doc_finalizer`(이슈 있으면 수정)

In [ ]:
# Corporate Doc Chatbot의 State Graph 구조 (원서 rag.py 발췌)
# 실행하려면: pip install langgraph langchain-groq, 그리고 GROQ_API_KEY 필요
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
    question: str
    context: List[Document]
    answer: str
    issues_report: str
    issues_detected: bool
    # messages: Annotated[list, add_messages]   # LangGraph 메시지 누적

# 그래프 골격 (의사코드):
graph_sketch = """
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver

graph_builder = StateGraph(State).add_sequence(
    [retrieve, generate, double_check, doc_finalizer]
)
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("doc_finalizer", END)
graph = graph_builder.compile(checkpointer=MemorySaver())
"""
print(graph_sketch)
print("4개 노드: retrieve -> generate -> double_check -> doc_finalizer")

In [ ]:
# Streamlit UI 골격 (원서 streamlit_app.py 발췌) — .py로 저장 후 실행
streamlit_sketch = """
import streamlit as st

st.set_page_config(page_title="Corporate Docs Chatbot", layout="wide")

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Chat Interface")
    if user_message := st.chat_input("Ask about the documents..."):
        response = process_message(user_message)
        st.chat_message("Assistant").markdown(response)

with col2:
    uploaded_files = st.file_uploader(
        "Upload", type=["pdf", "txt", "epub", "docx"],
        accept_multiple_files=True,
    )
"""
print(streamlit_sketch)
print("실행:  PYTHONPATH=. streamlit run chapter4/streamlit_app.py")

### 5.2 RAG 7가지 실패 지점 (Barnett et al., 2024)

| # | 실패 지점 | 원인 | 대응 |
|---|-----------|------|------|
| 1 | **Missing Content** | 관련 문서 자체가 없음 | 수집 시 검증 · 도메인 자료 보강 |
| 2 | **Missed Top Ranks** | 있는데 못 찾음 | 임베딩 모델 교체 · Hybrid · 문장 단위 검색 |
| 3 | **Context Window 초과** | 잘려서 누락 | Chunking 최적화 · 핵심 문장 추출 |
| 4 | **Extraction Failure** | LLM이 합성 실패 | 명시적 지시 + Contrastive 예시 |
| 5 | **Format 미준수** | JSON·표 형식 오류 | Output Parser · 후처리 검증 |
| 6 | **Specificity Mismatch** | 너무 일반적/상세 | 쿼리 확장 + 전문도 조정 |
| 7 | **Incomplete Info** | 일부만 답함 | MMR · 다양성 + 쿼리 변환 |

> **예방 원칙**: 수집 시 데이터 품질 검증 · 지속적 모니터링/캘리브레이션 · 사용자 피드백 반영 · KB 정기 갱신

---
## 📌 CHAPTER SUMMARY — 검색을 더한 생성

1. **RAG 기초** — 외부 지식으로 LLM 한계 보완. 4개 구성요소(KB·Retriever·Augmenter·Generator) + 2개 파이프라인(Indexing·Query).
2. **임베딩 · Vector Store** — Dense vector로 의미 검색. ChromaDB부터 Pinecone까지. HNSW가 프로덕션 표준, IVF+PQ로 메모리 절감.
3. **파이프라인 & Chunking** — Loader → Splitter(Recursive 권장) → Retriever. 청킹 전략이 검색 품질을 결정.
4. **Advanced RAG** — Hybrid · Re-rank · HyDE · MMR · Attribution · CRAG · Agentic. 요구사항에 맞춰 점진적으로 추가.

### 🧠 복습 문제 (Review Questions)
1. RAG에서 벡터 임베딩의 핵심 이점은 무엇인가?
2. MMR은 어떻게 문서 검색 결과를 개선하는가?
3. 왜 청킹이 효과적인 검색에 필수적인가?
4. Hybrid Search는 어떻게 검색 프로세스를 강화하는가?
5. Contextual Compression은 LLM 처리 전 정보를 어떻게 정제하나?
6. RAG에서 환각을 줄이기 위한 전략들은?

